# Section 2 — Loss Quantification & Expected Loss Modeling

**Course:** IDS 583 — Credit Risk Management
**Member A responsibility:** turn the 12-month PD output into Expected Loss and CECL-style lifetime reserves, then backtest against realized losses.

---

## Table of contents

### PART A — Loss Quantification (Member A's scope)

| # | Topic | What it does |
|---|---|---|
| A.1 | Setup & imports | Paths, constants, matplotlib config |
| A.2 | Data loading & merging | Load teammate's `loan_level_12m_outputs.csv` + pull reference fields |
| A.3 | LGD modeling | Derive LGD from charge-off data; rename to unified `LGD` |
| A.4 | EAD definition | Accept teammate's `ead_proxy = funded_amnt` |
| A.5 | Per-loan EL | `EL = PD × LGD × EAD` for every loan |
| A.6 | Portfolio aggregation | Headline numbers: total EL, loss rate, expected default count |
| A.7 | Segmentation | By grade, term, purpose, FICO band, + Grade × FICO cross-tab |
| A.8 | LGD sensitivity | Stress LGD anchors including the backtest-implied value |
| A.9 | Backtest | Vintage-level predicted EL vs realized loss (incl. survivorship-bias fix) |
| A.10 | CECL lifetime ECL | Extend 12m PD to 36/60-month term structure |
| A.11 | Macro overlay | Base / Adverse / Severely Adverse stress scenarios |
| A.12 | LGD distribution | Plot empirical loss rate on defaulted loans |
| A.13 | Export | Write loan-level and portfolio summary files |

### PART B — Pricing & Origination (Member B's scope)

> The cells in Part B were initially drafted here but are **Member B's responsibility** going forward. They are kept for continuity but will be moved to B's own notebook.

| # | Topic | Notes |
|---|---|---|
| B.1 | Required rate | Preliminary calc; Member B will extend with funding cost / capital charge |
| B.2 | Required vs actual int_rate | Scatter for B's risk-based pricing analysis |

### PART Z — Take-aways for the report narrative

---

## Modeling framework (Basel / CECL standard form)

$$\text{EL} = \text{PD} \times \text{LGD} \times \text{EAD}$$

- **PD** — 12-month probability of default (from teammate's discrete-time hazard model → `cum_pd_logit`)
- **LGD** — portfolio loss given default, derived from LendingClub's charge-off data (`recoveries`, `total_rec_prncp`)
- **EAD** — exposure at default, proxied by `funded_amnt` (teammate's convention)


## A.1 — Setup & imports

Load libraries, set display options, and define data paths. `LGD_CONST` is the portfolio-level LGD derived in Section 1 using the Basel/CECL "outstanding balance" denominator. To keep the LGD and EAD conventions aligned (textbook Basel/CECL framework), we also use **outstanding balance for EAD** (see A.4 — switched from `funded_amnt` to `funded_amnt − total_rec_prncp`). We treat LGD as a constant here and stress-test it in A.8.

In [ ]:
# A.1 — Setup
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Data paths — adjust if your teammate saved outputs elsewhere.
DATA_DIR = Path("/Users/dongshuxin/Desktop/Projects/data")
PROC_DIR = Path("/Users/dongshuxin/Desktop/Projects/data/processed")
OUT_DIR  = Path("/Users/dongshuxin/Desktop/Projects/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_ACCEPTED = DATA_DIR / "accepted_2007_to_2018Q4.csv.gz"
SURV_BASE    = PROC_DIR / "loan_level_survival_base.csv"
PD_OUTPUT    = PROC_DIR / "loan_level_12m_outputs.csv"

# Portfolio-level LGD constant from Section 1 (outstanding-balance convention)
LGD_CONST = 0.915423
print(f"LGD (Basel/CECL outstanding-balance convention) = {LGD_CONST:.4f}")
print("EAD convention: outstanding balance (set in A.4 as funded_amnt - total_rec_prncp)")

## A.2 — Data loading & merging

The teammate's `loan_level_12m_outputs.csv` contains the 12-month cumulative PD (`cum_pd_logit`) for every loan in the hazard-model sample (17,202 loans × 49 columns).

We pull three additional pieces of information that are essential for Member A's work but **not** in the teammate's output:

1. `term` (36 vs 60 months) and `fico_range_low` — from `loan_level_survival_base.csv`
2. `grade`, `sub_grade`, `int_rate` — from the raw LendingClub file
3. `loan_status`, `recoveries`, `total_rec_prncp`, `funded_amnt` — also from the raw file, used for the backtest in A.9

After merging, we rename the key columns to a **unified naming convention** (`PD_12m`, `LGD`, `EAD`, `EL_12m`) so downstream cells don't have to remember which logit variant the teammate used.

### A.2.1 — Load teammate's PD output

In [ ]:
# A.2.1 — Load teammate's loan-level 12-month PD output
print(f"Reading: {PD_OUTPUT}")
pd12 = pd.read_csv(PD_OUTPUT)
print(f"Shape: {pd12.shape}")
print(f"Key columns present:")
for col in ["id", "issue_d", "cum_pd_logit", "term_num", "event", "duration_months"]:
    print(f"  {col:<24} {'OK' if col in pd12.columns else 'MISSING'}")
pd12.head(3)

### A.2.2 — Merge `fico_range_low` from survival base

`loan_level_survival_base.csv` is the intermediate file produced during PD modeling. It carries the FICO credit-score bucket (`fico_range_low`) which we need for A.7.4 (FICO segmentation) and A.9 (backtest anchors).

In [ ]:
# A.2.2 — Join FICO band from survival_base
surv_cols = ["id", "fico_range_low"]
surv = pd.read_csv(SURV_BASE, usecols=surv_cols)
surv["id"] = pd.to_numeric(surv["id"], errors="coerce")
pd12["id"] = pd.to_numeric(pd12["id"], errors="coerce")

before = len(pd12)
pd12 = pd12.merge(surv, on="id", how="left")
assert len(pd12) == before, "Merge should be 1:1"
print(f"fico_range_low merged. Non-null: {pd12['fico_range_low'].notna().sum():,} / {len(pd12):,}")

### A.2.3 — Merge origination fields + EAD building blocks from raw file

From the raw LendingClub file (2.26M rows) we pull only the columns we actually need:

- `grade`, `sub_grade`, `int_rate` — origination-time fields used in A.7 (segmentation) and Part B (pricing).
- `funded_amnt`, `total_rec_prncp` — principal columns used in A.4 to build the outstanding-balance EAD, and reused in A.9 for the realized-loss backtest.

Pulling them here (rather than re-reading the raw file in A.9) avoids a second 2.26M-row scan.

In [ ]:
# A.2.3 — Join grade / sub_grade / int_rate / funded_amnt / total_rec_prncp from raw file
need = ["id", "grade", "sub_grade", "int_rate", "funded_amnt", "total_rec_prncp"]
raw = pd.read_csv(RAW_ACCEPTED, usecols=need, low_memory=False)
raw["id"] = pd.to_numeric(raw["id"], errors="coerce")

# Drop any columns that somehow already live in pd12
already = [c for c in raw.columns if c in pd12.columns and c != "id"]
if already:
    raw = raw.drop(columns=already)

before = len(pd12)
pd12 = pd12.merge(raw, on="id", how="left")
assert len(pd12) == before
for col in ["grade", "int_rate", "funded_amnt", "total_rec_prncp"]:
    n_nn = pd12[col].notna().sum()
    print(f"{col:<20} non-null: {n_nn:>7,} / {len(pd12):,}")

### A.2.4 — Unified naming convention

Rename the teammate's logit-specific columns to short, model-agnostic names so the rest of the notebook is readable.

In [ ]:
# A.2.4 — Unified naming (PD) + broadcast LGD
# Note: EAD is NOT set here anymore. A.4 builds it from
#       funded_amnt − total_rec_prncp (outstanding-balance convention).
pd12 = pd12.rename(columns={
    "cum_pd_logit": "PD_12m",
})
pd12["LGD"] = LGD_CONST   # broadcast constant LGD to every row

print("Unified columns in pd12 (EAD will be built in A.4):")
for col in ["PD_12m", "LGD"]:
    print(f"  {col:<12} min={pd12[col].min():.4f}  max={pd12[col].max():.4f}  mean={pd12[col].mean():.4f}")

### A.2.5 — Sanity check on PD

In [ ]:
# A.2.5 — PD sanity check
q = pd12["PD_12m"].quantile([0.01, 0.05, 0.50, 0.95, 0.99])
print("PD_12m percentiles:")
print(q.to_string())
print(f"\nPD_12m mean:     {pd12['PD_12m'].mean():.4%}")
print(f"PD_12m median:   {pd12['PD_12m'].median():.4%}")
print(f"Observed 12m default rate (event mean): {pd12['event'].mean():.4%}")

## A.3 — LGD modeling

In Section 1 we derived `LGD_CONST = 0.9154` using the **outstanding-balance denominator** (Basel/CECL convention):

$$\text{LGD}_{\text{outstanding}} = 1 - \frac{\text{recoveries}}{\text{principal outstanding at default}}$$

This is the industry-standard convention for unsecured consumer loans and will drive all downstream EL figures.

**Convention consistency:** because the LGD denominator is *principal outstanding at default*, we use the same denominator for EAD in A.4 (`EAD = funded_amnt − total_rec_prncp`). With this alignment, `LGD × EAD = realized_loss` for a defaulted loan, which is the textbook Basel/CECL identity. See A.8 for sensitivity analysis across alternative LGD anchors.

In [ ]:
# A.3 — LGD summary
# (Full derivation is in the Section 1 notebook; we just re-state the number here.)
print(f"Portfolio-level LGD (outstanding-balance convention): {LGD_CONST:.4%}")
print(f"Broadcast to {len(pd12):,} loans via pd12['LGD'] in A.2.4")

## A.4 — EAD definition (outstanding-balance convention)

For strict Basel/CECL consistency with `LGD_outstanding`, we define EAD as the **principal outstanding at the prediction time** — not `funded_amnt`. This requires one small calculation using two raw columns already on the dataframe:

$$\text{EAD}_i \;=\; \text{funded\_amnt}_i \;-\; \text{total\_rec\_prncp}_i$$

This single formula handles both cases:

- **Defaulted loans (`event = 1`):** `total_rec_prncp` is the principal repaid up to charge-off, so the subtraction gives the outstanding at default — exactly the Basel definition.
- **Surviving loans (`event = 0`):** `total_rec_prncp` is the principal repaid up to the data snapshot, so the subtraction gives the current outstanding — the appropriate starting balance for a forward-looking 12-month PD.

**Why this matters:** using `funded_amnt` as EAD (the teammate's original proxy) double-counts principal that has already been paid down. Pairing the inflated EAD with `LGD_outstanding = 0.9154` would over-state EL by ~43% on the 2017Q1 cohort — we demonstrate this alignment explicitly in the A.9 backtest.

In [ ]:
# A.4 — EAD under OUTSTANDING convention (Basel/CECL textbook)
# Previous: pd12['EAD'] = pd12['funded_amnt']  (via ead_proxy rename in A.2.4)
# Corrected: outstanding balance at prediction time
#   - defaulters: outstanding at default  = funded - total_rec_prncp_at_chargeoff
#   - survivors:  current outstanding     = funded - total_rec_prncp_at_snapshot
pd12['EAD'] = (pd12['funded_amnt'] - pd12['total_rec_prncp']).clip(lower=0)

# Compare to the previous funded-based EAD (for reference)
funded_total = pd12['funded_amnt'].sum()
ead_total    = pd12['EAD'].sum()
shrink_ratio = ead_total / funded_total

print(f"EAD convention: outstanding balance (funded_amnt - total_rec_prncp)")
print(f"  Total EAD (outstanding):      ${ead_total:,.0f}")
print(f"  Total funded_amnt (reference):${funded_total:,.0f}")
print(f"  Outstanding / funded ratio:   {shrink_ratio:.4f}")
print()
print(f"  Mean EAD per loan:   ${pd12['EAD'].mean():,.0f}")
print(f"  Median EAD per loan: ${pd12['EAD'].median():,.0f}")
print()
# Quick split by default status to show this behaves sensibly
for status, g in pd12.groupby("event"):
    label = "defaulters" if status == 1 else "survivors"
    r = (g["EAD"].sum() / g["funded_amnt"].sum())
    print(f"  {label:12s} (n={len(g):>5,}): outstanding/funded = {r:.4f}")

## A.5 — Per-loan Expected Loss

$$\text{EL}_i = \text{PD}_i \times \text{LGD} \times \text{EAD}_i$$

Because `LGD` is a portfolio constant in our baseline, all borrower-level variation in EL comes from PD and EAD. The `EL_rate` (EL / EAD) therefore moves 1-for-1 with `LGD × PD`.

In [ ]:
# A.5 — Per-loan EL
pd12["EL_12m"]   = pd12["PD_12m"] * pd12["LGD"] * pd12["EAD"]
pd12["EL_rate"]  = pd12["PD_12m"] * pd12["LGD"]   # independent of EAD

print(f"EL_12m  sum:     ${pd12['EL_12m'].sum():,.0f}")
print(f"EL_12m  mean:    ${pd12['EL_12m'].mean():,.2f}")
print(f"EL_rate mean:    {pd12['EL_rate'].mean():.4%}")
pd12[["id", "PD_12m", "LGD", "EAD", "EL_12m", "EL_rate"]].head()

## A.6 — Portfolio aggregation

Headline numbers we will put in the executive summary:
- total exposure (`sum(EAD)`)
- total 12-month expected loss (`sum(EL_12m)`)
- portfolio loss rate (`sum(EL) / sum(EAD)`)
- implied default count (`sum(PD × EAD) / sum(EAD)` = EAD-weighted PD)

In [ ]:
# A.6 — Portfolio headline
total_ead       = pd12["EAD"].sum()
total_el        = pd12["EL_12m"].sum()
portfolio_rate  = total_el / total_ead
ead_weighted_pd = np.average(pd12["PD_12m"], weights=pd12["EAD"])
expected_defs   = (pd12["PD_12m"] * pd12["EAD"]).sum() / pd12["EAD"].mean()  # dollar-weighted
n_expected_defs_naive = pd12["PD_12m"].sum()  # loan-count expectation

print(f"Loans in sample:        {len(pd12):,}")
print(f"Total EAD:              ${total_ead:,.0f}")
print(f"Total 12m EL:           ${total_el:,.0f}")
print(f"Portfolio 12m loss rate:{portfolio_rate:.4%}")
print(f"EAD-weighted PD:        {ead_weighted_pd:.4%}")
print(f"Expected # defaults (loan count basis): {n_expected_defs_naive:.0f}")

## A.7 — Segmentation analysis

Where does the loss come from? We slice the portfolio by five dimensions:
- **Grade** (LC's proprietary A-G grade, assigned at origination)
- **Term** (36 vs 60 months)
- **Purpose** (debt consolidation, credit card, small business, ...)
- **FICO band** (external credit-score buckets)
- **Grade × FICO cross-tab** — reveals whether the two signals carry redundant or complementary information

### A.7.1 — By grade

Expected finding: EL concentrates in low-grade buckets (D/E/F/G) even though they hold a smaller EAD share.

In [ ]:
# A.7.1 — Segment EL by grade
if "grade" in pd12.columns and pd12["grade"].notna().any():
    grade_order = ["A", "B", "C", "D", "E", "F", "G"]
    seg_grade = (pd12.groupby("grade", observed=True)
                    .agg(loans=("EAD", "size"),
                         ead=("EAD", "sum"),
                         pd_w=("PD_12m", lambda s: np.average(s, weights=pd12.loc[s.index, "EAD"])),
                         el=("EL_12m", "sum"))
                    .reindex(grade_order)
                    .assign(loss_rate=lambda d: d["el"] / d["ead"],
                            ead_share=lambda d: d["ead"] / d["ead"].sum(),
                            el_share=lambda d: d["el"] / d["el"].sum()))
    print(seg_grade[["loans", "ead", "ead_share", "pd_w", "el", "el_share", "loss_rate"]].round(4))
else:
    print("grade column missing or all-NaN")

In [ ]:
# A.7.1b — Visual: EAD share vs EL share by grade (classic concentration chart)
if "grade" in pd12.columns and pd12["grade"].notna().any():
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(seg_grade.index))
    w = 0.38
    ax.bar(x - w/2, seg_grade["ead_share"], w, label="EAD share")
    ax.bar(x + w/2, seg_grade["el_share"],  w, label="EL share")
    ax.set_xticks(x)
    ax.set_xticklabels(seg_grade.index)
    ax.set_ylabel("Share of portfolio")
    ax.set_title("EAD share vs EL share by grade — loss concentrates in low grades")
    ax.legend()
    plt.tight_layout()
    plt.show()

### A.7.2 — By term (36 vs 60 months)

Expected finding: 60-month loans have meaningfully higher 12m loss rates because they are self-selected toward lower-credit / higher-risk borrowers (LC charges more for 60m).

In [ ]:
# A.7.2 — Segment by term
seg_term = (pd12.groupby("term_num", observed=True)
               .agg(loans=("EAD", "size"),
                    ead=("EAD", "sum"),
                    pd_w=("PD_12m", lambda s: np.average(s, weights=pd12.loc[s.index, "EAD"])),
                    el=("EL_12m", "sum"))
               .assign(loss_rate=lambda d: d["el"] / d["ead"]))
print(seg_term.round(4))

In [ ]:
# A.7.2a — 36m vs 60m comparison chart
terms       = seg_term.index.astype(int).tolist()
loss_rates  = seg_term["loss_rate"].values * 100
pd_weights  = seg_term["pd_w"].values * 100
ead_shares  = (seg_term["ead"] / seg_term["ead"].sum() * 100).values

fig, ax = plt.subplots(figsize=(8.5, 5.5))
x = np.arange(len(terms))
w = 0.38

ax.bar(x - w/2, loss_rates, w, label="12m loss rate",
       color="#d62728", edgecolor="black", linewidth=0.5)
ax.bar(x + w/2, pd_weights, w, label="EAD-weighted PD",
       color="#ff9896", edgecolor="black", linewidth=0.5)

for i, (lr, pdw) in enumerate(zip(loss_rates, pd_weights)):
    ax.text(i - w/2, lr + 0.08, f"{lr:.2f}%", ha="center",
            fontsize=10, fontweight="bold")
    ax.text(i + w/2, pdw + 0.08, f"{pdw:.2f}%", ha="center",
            fontsize=10, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels([f"{t} months\n(EAD share: {es:.1f}%)"
                    for t, es in zip(terms, ead_shares)])
ax.set_ylabel("Rate (%)")
ax.set_title("36-month vs 60-month loans\n"
             "60m borrowers self-select to higher risk")
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, max(loss_rates.max(), pd_weights.max()) * 1.25)
plt.tight_layout()
plt.savefig(OUT_DIR / "term_36_vs_60.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR / 'term_36_vs_60.png'}")

### A.7.3 — By purpose (top 8)

Expected finding: `small_business` and `moving` show elevated loss rates; `debt_consolidation` dominates EAD share but has middle-of-the-pack loss rates.

In [ ]:
# A.7.3 — Segment by purpose
seg_purp = (pd12.groupby("purpose", observed=True)
               .agg(loans=("EAD", "size"),
                    ead=("EAD", "sum"),
                    pd_w=("PD_12m", lambda s: np.average(s, weights=pd12.loc[s.index, "EAD"])),
                    el=("EL_12m", "sum"))
               .assign(loss_rate=lambda d: d["el"] / d["ead"])
               .sort_values("ead", ascending=False)
               .head(8))
print(seg_purp.round(4))

In [ ]:
# A.7.3a — Horizontal bar chart of loss rate by purpose
seg_purp_sorted = seg_purp.sort_values("loss_rate", ascending=True)
loss_rates_pct  = seg_purp_sorted["loss_rate"].values * 100
ead_shares      = (seg_purp_sorted["ead"] / seg_purp_sorted["ead"].sum() * 100).values
purposes        = seg_purp_sorted.index.tolist()

norm   = loss_rates_pct / loss_rates_pct.max()
colors = plt.cm.RdYlGn_r(0.3 + 0.6 * norm)

fig, ax = plt.subplots(figsize=(11, 6))
y = np.arange(len(purposes))
ax.barh(y, loss_rates_pct, color=colors, edgecolor="black", linewidth=0.5)

for i, (r, s) in enumerate(zip(loss_rates_pct, ead_shares)):
    ax.text(r + 0.08, i, f"{r:.2f}%   |  EAD share: {s:.1f}%",
            va="center", fontsize=9)

ax.set_yticks(y)
ax.set_yticklabels(purposes)
ax.set_xlabel("12-month loss rate (%)")
ax.set_title("Loss rate by loan purpose\n"
             "small_business is the concentration-limit candidate")
ax.grid(axis="x", alpha=0.3)
ax.set_xlim(0, max(loss_rates_pct) * 1.35)
plt.tight_layout()
plt.savefig(OUT_DIR / "loss_rate_by_purpose.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR / 'loss_rate_by_purpose.png'}")

### A.7.4 — By FICO band

FICO is an external credit score (300-850) published by Fair Isaac Corporation. It is the most widely used credit-risk signal in US consumer lending and is **complementary** to LC's internal grade:

| FICO | Market label |
|---|---|
| < 660 | subprime |
| 660-700 | near-prime |
| 700-740 | prime |
| 740-780 | super-prime |
| 780+ | top tier |

Expected finding: monotonic improvement in loss rate as FICO rises.

In [ ]:
# A.7.4 — Segment by FICO band
if "fico_range_low" in pd12.columns:
    bins   = [0, 660, 700, 740, 780, 850]
    labels = ["<660 (subprime)", "660-700 (near-prime)", "700-740 (prime)",
              "740-780 (super-prime)", "780+ (top tier)"]
    pd12["fico_band"] = pd.cut(pd12["fico_range_low"], bins=bins, labels=labels, right=False)
    seg_fico = (pd12.groupby("fico_band", observed=True)
                   .agg(loans=("EAD", "size"),
                        ead=("EAD", "sum"),
                        pd_w=("PD_12m", lambda s: np.average(s, weights=pd12.loc[s.index, "EAD"])),
                        el=("EL_12m", "sum"))
                   .assign(loss_rate=lambda d: d["el"] / d["ead"]))
    print(seg_fico.round(4))
else:
    print("fico_range_low missing — skipping FICO segmentation")

In [ ]:
# A.7.4a — Loss rate by FICO band (monotonic bar)
if "fico_band" in pd12.columns:
    fico_bands  = seg_fico.index.tolist()
    loss_rates  = seg_fico["loss_rate"].values * 100
    counts      = seg_fico["loans"].values

    colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(fico_bands)))

    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.bar(range(len(fico_bands)), loss_rates,
           color=colors, edgecolor="black", linewidth=0.5)

    for i, (r, n) in enumerate(zip(loss_rates, counts)):
        ax.text(i, r + 0.1, f"{r:.2f}%\nn={n:,}",
                ha="center", fontsize=9, fontweight="bold")

    ax.set_xticks(range(len(fico_bands)))
    ax.set_xticklabels(fico_bands, rotation=15, ha="right")
    ax.set_ylabel("12-month loss rate (%)")
    ax.set_title("Loss rate by FICO band\n"
                 "Monotonic improvement from subprime to top tier")
    ax.grid(axis="y", alpha=0.3)
    ax.set_ylim(0, max(loss_rates) * 1.30)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "loss_rate_by_fico.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {OUT_DIR / 'loss_rate_by_fico.png'}")

### A.7.5 — Grade × FICO cross-tab (loss-rate heatmap)

If Grade and FICO were redundant, the heatmap would look like a diagonal gradient. If they are **complementary**, within every grade the loss rate will still improve materially as FICO rises — which means the two signals carry independent information.

In [ ]:
# A.7.5 — Grade × FICO cross-tab of loss rate
if "grade" in pd12.columns and "fico_band" in pd12.columns:
    # Build loss-rate pivot by summing EL and EAD
    el_pv  = pd12.pivot_table(index="grade", columns="fico_band",
                              values="EL_12m", aggfunc="sum",
                              observed=True).reindex(grade_order)
    ead_pv = pd12.pivot_table(index="grade", columns="fico_band",
                              values="EAD", aggfunc="sum",
                              observed=True).reindex(grade_order)
    loss_rate_pv = (el_pv / ead_pv).round(4)

    # Also build a loan-count pivot to flag thin cells
    cnt_pv = pd12.pivot_table(index="grade", columns="fico_band",
                              values="id", aggfunc="count",
                              observed=True).reindex(grade_order)

    print("Loss rate — Grade × FICO:")
    print(loss_rate_pv)
    print()
    print("Loan count — Grade × FICO:")
    print(cnt_pv.fillna(0).astype(int))

#### A.7.5a — Heatmap visualization

A single chart for the presentation: colour intensity = loss rate, annotations = loss rate % + loan count. White cells are too thin to draw conclusions from (< 20 loans).

In [ ]:
# A.7.5a — Grade × FICO heatmap
if "grade" in pd12.columns and "fico_band" in pd12.columns:
    import matplotlib.colors as mcolors
    data  = loss_rate_pv.values.astype(float) * 100    # convert to percentage
    counts = cnt_pv.reindex_like(loss_rate_pv).fillna(0).values

    fig, ax = plt.subplots(figsize=(11, 6))
    # Mask NaN cells
    masked = np.ma.masked_invalid(data)
    cmap = plt.cm.YlOrRd.copy()
    cmap.set_bad(color="#eeeeee")
    vmax = np.nanmax(data)
    im = ax.imshow(masked, cmap=cmap, vmin=0, vmax=vmax, aspect="auto")

    # Axis ticks and labels
    ax.set_xticks(np.arange(len(loss_rate_pv.columns)))
    ax.set_xticklabels(loss_rate_pv.columns, rotation=20, ha="right")
    ax.set_yticks(np.arange(len(loss_rate_pv.index)))
    ax.set_yticklabels(loss_rate_pv.index)
    ax.set_xlabel("FICO band")
    ax.set_ylabel("LC grade")
    ax.set_title("12-month loss rate — Grade × FICO\n(colour = loss rate %, annotation = loss % / loan count)")

    # Annotate cells
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            n = int(counts[i, j])
            if np.isnan(v) or n == 0:
                txt = "n/a"
                color = "gray"
            else:
                thin = n < 20
                txt = f"{v:.2f}%\nn={n:,}"
                # choose text colour based on background intensity
                color = "white" if v > vmax * 0.55 else "black"
                if thin:
                    txt += "*"   # mark thin cells
            ax.text(j, i, txt, ha="center", va="center",
                    fontsize=9, color=color, fontweight="bold" if not np.isnan(v) else "normal")

    # Colorbar
    cbar = fig.colorbar(im, ax=ax, shrink=0.85)
    cbar.set_label("12m loss rate (%)")
    # Footer note
    fig.text(0.01, -0.02, "* cell has fewer than 20 loans — treat with caution.",
             fontsize=8, style="italic", color="gray")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "grade_x_fico_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {OUT_DIR / 'grade_x_fico_heatmap.png'}")

**How to read this chart in the presentation:**

1. **Row-wise gradient (left → right):** within every grade, the loss rate drops as FICO rises. That's the complementary-information story — LC's internal grade does not fully exploit FICO.
2. **Column-wise gradient (A → G):** within every FICO band, the loss rate rises as grade deteriorates. That's the classic grade-ordering story — LC's grade is doing real work even after controlling for FICO.
3. **Off-diagonal cells:** compare a "Grade A, 660-700" cell vs a "Grade D, 780+" cell. If the second is safer, you have a direct "pricing-arbitrage" hook for Member B's swap-set analysis.
4. **n/a or thin cells:** regions where very few loans sit. LC essentially never originates, for example, a Grade A loan to a sub-660 FICO borrower — the screening already eliminates that corner. Don't over-interpret those cells.

## A.8 — LGD sensitivity

Re-compute total portfolio EL under four economically-grounded LGD anchors. Since we now use `EAD_outstanding` (see A.4), all four anchors are applied to the **same outstanding-balance exposure** — no convention mismatch:

| Anchor | Value | Source |
|---|---|---|
| Section-1 median LGD_outstanding | **0.9154** | Basel/CECL convention (baseline) |
| Backtest-implied LGD_outstanding | ~0.91 | 2017Q1 defaulters' actual `realized_loss / outstanding_at_default` (A.9.5) — should closely match the baseline |
| Regulatory floor for unsecured consumer | 0.45 | Basel III LGD floor |
| Optimistic / recovery-heavy | 0.30 | Hypothetical (well-collateralized scenario) |

The spread between these four values shows **how much EL depends on the LGD assumption** — which is the largest remaining source of model uncertainty for an unsecured consumer portfolio now that the EAD/LGD convention mismatch has been resolved.

In [ ]:
# A.8 — LGD sensitivity table (all anchors applied to outstanding-balance EAD)
LGD_ANCHORS = {
    "Section-1 median LGD_outstanding (Basel/CECL baseline)" : LGD_CONST,  # 0.9154
    "Basel III unsecured regulatory floor"                   : 0.45,
    "Optimistic / recovery-heavy"                            : 0.30,
}

rows = []
for name, lgd_val in LGD_ANCHORS.items():
    el_scen = (pd12["PD_12m"] * lgd_val * pd12["EAD"]).sum()
    rows.append({
        "LGD anchor": name,
        "LGD": f"{lgd_val:.4f}",
        "Total EL (USD)": f"${el_scen:,.0f}",
        "Loss rate": f"{el_scen / total_ead:.4%}",
        "vs baseline": f"{el_scen / total_el:.2f}x",
    })
sens_df = pd.DataFrame(rows)
print(sens_df.to_string(index=False))

## A.9 — Backtest / vintage analysis

Industry practice: compare **predicted EL** against **realized loss** by origination cohort (vintage).

For a loan with `event == 1` (defaulted within 12 months of origination in our hazard window), realized loss is:

$$\text{realized\_loss}_i = \max\bigl(\text{funded\_amnt}_i - \text{total\_rec\_prncp}_i - \text{recoveries}_i,\; 0\bigr)$$

If `event == 0`, realized loss is 0 (within the 12-month window).

### Backtest pitfalls (both surfaced in our analysis)

1. **Right-censoring** — vintages issued close to the dataset cutoff haven't had time to fully mature. Their observed default rate is artificially low. We handle this by focusing on the **fully-observed 2017Q1 cohort** in A.9.4.

2. **Survivorship bias** — an earlier attempt filtered on `duration_months >= 24` to find "mature" loans. This filter **excludes all defaulters** (whose `duration_months` = time-to-default, typically < 12) and introduces a massive bias toward survivors. The correct filter is **by origination vintage**, not observed tenure.

### A.9.1 — Pull actual loss fields

In [ ]:
# A.9.1 — Pull remaining loss fields from the raw file
#   (funded_amnt + total_rec_prncp were already merged in A.2.3)
need = ["id", "loan_status", "recoveries"]
raw_bt = pd.read_csv(RAW_ACCEPTED, usecols=need, low_memory=False)
raw_bt["id"] = pd.to_numeric(raw_bt["id"], errors="coerce")

already = [c for c in raw_bt.columns if c in pd12.columns and c != "id"]
if already:
    raw_bt = raw_bt.drop(columns=already)

before = len(pd12)
pd12 = pd12.merge(raw_bt, on="id", how="left")
assert len(pd12) == before

print("loan_status distribution (top 10):")
print(pd12["loan_status"].value_counts().head(10))

### A.9.2 — Compute realized loss

In [ ]:
# A.9.2 — Realized loss
pd12["realized_loss"] = np.where(
    pd12["event"] == 1,
    (pd12["funded_amnt"]
        - pd12["total_rec_prncp"].fillna(0)
        - pd12["recoveries"].fillna(0)
    ).clip(lower=0),
    0.0,
)

total_realized = pd12["realized_loss"].sum()
n_defaults     = int(pd12["event"].sum())
print(f"# defaults (event=1):      {n_defaults:,}")
print(f"Realized default rate:     {pd12['event'].mean():.4%}")
print(f"\nTotal realized loss:       ${total_realized:,.0f}")
print(f"Total predicted EL_12m:    ${total_el:,.0f}")
print(f"Coverage ratio:            {total_el/total_realized:.2f}x")
print(f"Over-reserve vs realized:  {(total_el/total_realized - 1)*100:+.1f}%")

### A.9.3 — Vintage-level comparison

Group by quarterly origination vintage and compare predicted EL to realized loss. Right-censoring will make newer vintages look over-reserved.

In [ ]:
# A.9.3 — Vintage backtest
pd12["issue_d"] = pd.to_datetime(pd12["issue_d"], errors="coerce")
pd12["vintage"] = pd12["issue_d"].dt.to_period("Q").astype(str)

vintage = (pd12.groupby("vintage")
              .agg(loans=("id", "count"),
                   ead=("EAD", "sum"),
                   pred_pd=("PD_12m", lambda s: np.average(s, weights=pd12.loc[s.index, "EAD"])),
                   actual_pd=("event", lambda s: np.average(s, weights=pd12.loc[s.index, "EAD"])),
                   pred_el=("EL_12m", "sum"),
                   actual_loss=("realized_loss", "sum"))
              .assign(pd_gap=lambda d: d["pred_pd"] - d["actual_pd"],
                      el_error_pct=lambda d: np.where(d["actual_loss"] > 0,
                                                       d["pred_el"] / d["actual_loss"] - 1,
                                                       np.nan)))
print(vintage.round(4))

In [ ]:
# A.9.3a — Vintage backtest visualization
fig, ax = plt.subplots(figsize=(12, 6))
vintages_list = vintage.index.tolist()
x = np.arange(len(vintages_list))
w = 0.38
fully_observed = "2017Q1"

pred_colors = ["#1f77b4" if v == fully_observed else "#aec7e8" for v in vintages_list]
act_colors  = ["#d62728" if v == fully_observed else "#ff9896" for v in vintages_list]

ax.bar(x - w/2, vintage["pred_el"]/1e6,     w, label="Predicted EL",
       color=pred_colors, edgecolor="black", linewidth=0.5)
ax.bar(x + w/2, vintage["actual_loss"]/1e6, w, label="Realized Loss",
       color=act_colors, edgecolor="black", linewidth=0.5)

ymax = 0
for i, v in enumerate(vintages_list):
    pred = vintage.loc[v, "pred_el"] / 1e6
    act  = vintage.loc[v, "actual_loss"] / 1e6
    ymax = max(ymax, pred, act)
    if act > 0:
        ratio = pred / act
        ax.text(i, max(pred, act) + 0.08, f"{ratio:.1f}x",
                ha="center", fontsize=10, fontweight="bold",
                color="darkgreen" if v == fully_observed else "gray")

if fully_observed in vintages_list:
    idx_q1 = vintages_list.index(fully_observed)
    ax.axvspan(idx_q1 - 0.5, idx_q1 + 0.5, alpha=0.12, color="green", zorder=0)
    ax.text(idx_q1, ymax * 1.08, "Fully observed\n(ground truth)",
            ha="center", fontsize=9, color="darkgreen", style="italic")

ax.set_xticks(x)
ax.set_xticklabels(vintages_list)
ax.set_ylabel("USD (millions)")
ax.set_title("Vintage Backtest — Predicted EL vs Realized Loss\n"
             "Right-censoring inflates apparent over-reserve for newer vintages;\n"
             "the only fully-observed cohort (2017Q1) is calibrated at 1.13x")
ax.legend(loc="upper left")
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, ymax * 1.25)
plt.tight_layout()
plt.savefig(OUT_DIR / "vintage_backtest.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR / 'vintage_backtest.png'}")

### A.9.4 — Restricted backtest on fully-observed vintage (2017Q1)

Of all the vintages in our sample, **2017Q1** is the only one with a full 12-month observation window completed for every loan before the dataset cutoff. This is the apples-to-apples benchmark for the model's calibration.

In [ ]:
# A.9.4 — Restricted backtest: only 2017Q1 (fully observed)
q1 = pd12[pd12["vintage"] == "2017Q1"].copy()
print(f"2017Q1 cohort:  {len(q1):,} loans ({len(q1)/len(pd12):.1%} of total)")
print(f"Defaults:       {int(q1['event'].sum()):,} ({q1['event'].mean():.2%})")
print()
print(f"Predicted EL:   ${q1['EL_12m'].sum():,.0f}")
print(f"Realized loss:  ${q1['realized_loss'].sum():,.0f}")
print(f"Coverage ratio: {q1['EL_12m'].sum() / q1['realized_loss'].sum():.2f}x")
print(f"Error vs actual:{(q1['EL_12m'].sum() / q1['realized_loss'].sum() - 1)*100:+.1f}%")
print()
print(f"==> A coverage ratio within 0.85–1.15x is considered well-calibrated under CECL.")

### A.9.5 — Sanity check: empirical LGD_outstanding on 2017Q1

With the EAD convention switched to outstanding (A.4), we can verify our assumed `LGD_CONST = 0.9154` against the empirical value observed for 2017Q1 defaulters:

$$\text{LGD}_{\text{outstanding}}^{\text{empirical}} \;=\; \frac{\sum_{i \in \text{defaulters}} \text{realized\_loss}_i}{\sum_{i \in \text{defaulters}} \text{outstanding}_i}$$

where `outstanding_i = funded_amnt_i − total_rec_prncp_i` (same formula used in A.4). If our Section-1 LGD is correct, this ratio should land near **0.9154**. Any material deviation would suggest the Section-1 LGD needs re-estimation.

In [ ]:
# A.9.5 — Empirical LGD_outstanding on 2017Q1 defaulters (sanity check)
defaulters_q1 = q1[q1["event"] == 1].copy()

# Outstanding at default = funded - principal already repaid
defaulters_q1["outstanding_at_default"] = (
    defaulters_q1["funded_amnt"] - defaulters_q1["total_rec_prncp"]
).clip(lower=0)

empirical_lgd_outstanding = (
    defaulters_q1["realized_loss"].sum()
    / defaulters_q1["outstanding_at_default"].sum()
)

print(f"Defaulters in 2017Q1:           {len(defaulters_q1):,}")
print(f"Sum realized_loss:              ${defaulters_q1['realized_loss'].sum():,.0f}")
print(f"Sum outstanding at default:     ${defaulters_q1['outstanding_at_default'].sum():,.0f}")
print()
print(f"Empirical LGD_outstanding:      {empirical_lgd_outstanding:.4%}")
print(f"Section-1 LGD_CONST used:       {LGD_CONST:.4%}")
diff = empirical_lgd_outstanding - LGD_CONST
pct = diff / LGD_CONST
print(f"Difference:                     {diff:+.4%}  ({pct:+.2%} of LGD_CONST)")
print()
verdict = "✓ consistent" if abs(pct) < 0.10 else "⚠ review — Section-1 LGD may need re-estimation"
print(f"Verdict: {verdict}")

### A.9.6 — Interpretation: a clean PD-calibration read

With EAD and LGD both on the **outstanding-balance convention**, the 2017Q1 backtest gives us a clean read on PD calibration — there is no longer any hidden mismatch between the LGD denominator and the EAD denominator.

The interpretation is straightforward:

- If **coverage ratio ≈ 1.0x** → PD model is well-calibrated for this cohort.
- If **coverage ratio < 0.9x** → PD model **under-predicts** defaults; EL is too low → we are *under-reserved*. A PD recalibration (or a model-risk overlay) is required before production use.
- If **coverage ratio > 1.1x** → PD model **over-predicts** defaults; we are over-reserving.

**Previously observed (funded-EAD convention):** 1.13x looked well-calibrated only because two offsetting errors (LGD over-application × PD under-prediction) were cancelling each other out. With the convention fixed, the PD signal stands alone.

See the 2-bar chart below for the visual. Numerical details:

In [ ]:
# A.9.6 — Clean Predicted vs Realized summary (outstanding convention)
pd_pred_q1    = np.average(q1["PD_12m"], weights=q1["EAD"])
pd_actual_q1  = q1["event"].mean()
pd_gap_pct    = (pd_actual_q1 - pd_pred_q1) / pd_pred_q1

predicted_el  = q1["EL_12m"].sum()
realized_loss = q1["realized_loss"].sum()
coverage      = predicted_el / realized_loss

print("=" * 60)
print("2017Q1 backtest — outstanding-balance convention")
print("=" * 60)
print(f"Predicted PD (EAD-weighted): {pd_pred_q1:.4%}")
print(f"Actual PD (event rate):      {pd_actual_q1:.4%}")
print(f"PD gap:                      {pd_gap_pct:+.2%}  "
      f"({'under-predicted' if pd_gap_pct > 0 else 'over-predicted'})")
print()
print(f"Predicted EL:  ${predicted_el:,.0f}")
print(f"Realized loss: ${realized_loss:,.0f}")
print(f"Coverage:      {coverage:.2f}x")
print()
if coverage < 0.9:
    print("⚠  Under-reserved — recommend either PD recalibration\n"
          "   or a model-risk overlay of at least "
          f"{max(0, 1.0 - coverage):.0%} on top of EL.")
elif coverage > 1.1:
    print("⚠  Over-reserved — model is conservative.")
else:
    print("✓  Well-calibrated under CECL (0.9 – 1.1x).")

In [ ]:
# A.9.6a — 2-bar Predicted vs Realized chart (2017Q1, outstanding convention)
pred_m = predicted_el / 1e6
real_m = realized_loss / 1e6

labels = ["Predicted EL\n(PD × LGD_out × EAD_out)", "Realized loss\n(2017Q1 actual)"]
values = [pred_m, real_m]
colors = ["#1f77b4", "#d62728"]

fig, ax = plt.subplots(figsize=(8.5, 5.5))
bars = ax.bar(range(len(labels)), values, color=colors,
              edgecolor="black", linewidth=0.5, width=0.55)

for i, v in enumerate(values):
    ax.text(i, v + max(values) * 0.025, f"${v:.2f}M",
            ha="center", fontsize=12, fontweight="bold")

gap_arrow_y = max(values) * 1.1
ax.annotate("",
            xy=(1, gap_arrow_y), xytext=(0, gap_arrow_y),
            arrowprops=dict(arrowstyle="<->", color="gray"))
ax.text(0.5, gap_arrow_y * 1.03,
        f"Coverage: {coverage:.2f}x  |  PD gap: {pd_gap_pct:+.1%}",
        ha="center", fontsize=11,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray"))

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel("EL / Loss (USD millions)")
ax.set_title("2017Q1 backtest — clean PD calibration view\n"
             "(EAD and LGD both on outstanding-balance convention)")
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, max(values) * 1.25)
plt.tight_layout()
plt.savefig(OUT_DIR / "backtest_predicted_vs_realized.png",
            dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIR / 'backtest_predicted_vs_realized.png'}")

## A.10 — CECL lifetime ECL

CECL (Current Expected Credit Loss, ASC 326) requires reserving for the **lifetime** expected loss — not just 12 months. We extend the 12m cumulative PD to each loan's **full contractual term** (36 or 60 months).

### Method: constant-hazard extrapolation

From the discrete-time hazard model, the 12-month survival probability is $S(12) = 1 - \text{PD}_{12m}$.

If we assume a **constant monthly hazard** $h$, then $S(12) = (1-h)^{12}$, so:

$$h = 1 - (1 - \text{PD}_{12m})^{1/12}$$

Lifetime PD is then:

$$\text{PD}_{\text{lifetime}} = 1 - (1 - h)^{T}$$

where $T$ is the contractual term (36 or 60 months). This is equivalent to:

$$\text{PD}_{\text{lifetime}} = 1 - (1 - \text{PD}_{12m})^{T/12}$$

### Limitations

1. **Constant-hazard assumption:** Real consumer-loan hazard curves are non-constant — hazards peak around months 12-24 (post-honeymoon) and taper thereafter. Constant-hazard extrapolation is **conservative** (overstates) if the true curve tapers after month 12. A reasonable first cut for a student project; industrial models use vintage-level seasoning curves.
2. **EAD assumption:** We use **current outstanding balance** as a conservative proxy for lifetime EAD. In strict CECL, the lifetime EAD should be the *average remaining outstanding* over the loan's remaining contractual term — which for a standard amortizing loan is ~50-70% of current outstanding. Our approach therefore **over-states** lifetime ECL by ~30-50%, making it a prudent upper bound. A refined version would require projecting the full amortization schedule loan-by-loan.

We flag both limitations in the report.

In [ ]:
# A.10.1 — Monthly hazard and lifetime PD
pd12["monthly_hazard"] = 1 - (1 - pd12["PD_12m"]).clip(lower=1e-9)**(1/12)
pd12["lifetime_PD"]    = 1 - (1 - pd12["PD_12m"]).clip(lower=1e-9)**(pd12["term_num"]/12)

print("Monthly hazard percentiles:")
print(pd12["monthly_hazard"].quantile([0.05, 0.5, 0.95]).round(6).to_string())
print()
print("Lifetime PD by term:")
print(pd12.groupby("term_num")["lifetime_PD"].agg(["mean", "median", "max"]).round(4))

In [ ]:
# A.10.2 — Lifetime EL per loan
pd12["lifetime_EL"] = pd12["lifetime_PD"] * pd12["LGD"] * pd12["EAD"]

total_lifetime_el = pd12["lifetime_EL"].sum()
lifetime_rate     = total_lifetime_el / total_ead
mult_vs_12m       = total_lifetime_el / total_el

print(f"12-month EL:       ${total_el:,.0f}  (rate {total_el/total_ead:.4%})")
print(f"Lifetime ECL:      ${total_lifetime_el:,.0f}  (rate {lifetime_rate:.4%})")
print(f"Lifetime / 12m:    {mult_vs_12m:.2f}x")
print()
print("Breakdown by term:")
lt_by_term = (pd12.groupby("term_num")
                 .agg(loans=("EAD", "size"),
                      ead=("EAD", "sum"),
                      el_12m=("EL_12m", "sum"),
                      el_life=("lifetime_EL", "sum"))
                 .assign(lt_over_12m=lambda d: d["el_life"] / d["el_12m"]))
print(lt_by_term.round(2))

In [ ]:
# A.10.3 — Lifetime EL by grade (where does the reserve concentrate?)
if "grade" in pd12.columns and pd12["grade"].notna().any():
    seg_grade_lt = (pd12.groupby("grade", observed=True)
                       .agg(loans=("EAD", "size"),
                            ead=("EAD", "sum"),
                            el_12m=("EL_12m", "sum"),
                            el_life=("lifetime_EL", "sum"))
                       .reindex(grade_order)
                       .assign(loss_rate_12m=lambda d: d["el_12m"] / d["ead"],
                               loss_rate_life=lambda d: d["el_life"] / d["ead"],
                               ratio=lambda d: d["el_life"] / d["el_12m"]))
    print(seg_grade_lt[["loans", "ead", "el_12m", "el_life", "loss_rate_12m", "loss_rate_life", "ratio"]].round(4))

In [ ]:
# A.10.4 — Lifetime ECL vs 12m EL by grade (dual axis: dollars + multiplier)
if "grade" in pd12.columns and pd12["grade"].notna().any():
    seg_lt = seg_grade_lt
    fig, ax1 = plt.subplots(figsize=(11, 6))

    x = np.arange(len(seg_lt.index))
    w = 0.38
    ax1.bar(x - w/2, seg_lt["el_12m"] / 1e6,  w, label="12m EL",
            color="#1f77b4", edgecolor="black", linewidth=0.5)
    ax1.bar(x + w/2, seg_lt["el_life"] / 1e6, w, label="Lifetime ECL",
            color="#ff7f0e", edgecolor="black", linewidth=0.5)

    ax1.set_xticks(x)
    ax1.set_xticklabels(seg_lt.index)
    ax1.set_xlabel("LC grade")
    ax1.set_ylabel("Dollar EL / ECL (USD millions)")
    ax1.grid(axis="y", alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(x, seg_lt["ratio"], "r-o", label="Lifetime / 12m multiplier",
             linewidth=2, markersize=8)
    ax2.set_ylabel("Lifetime / 12m multiplier", color="red")
    ax2.tick_params(axis="y", labelcolor="red")
    for i, r in enumerate(seg_lt["ratio"]):
        ax2.text(i, r + 0.1, f"{r:.1f}x", ha="center", color="red",
                 fontsize=10, fontweight="bold")

    ax1.set_title("12-month EL vs Lifetime ECL by grade\n"
                  "CECL requires reserving for the full contractual term — "
                  "low grades compound fastest")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    plt.tight_layout()
    plt.savefig(OUT_DIR / "lifetime_vs_12m_by_grade.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {OUT_DIR / 'lifetime_vs_12m_by_grade.png'}")

## A.11 — Macro overlay / stress testing

CECL encourages incorporating **forward-looking information** into the reserve. We overlay three macroeconomic scenarios onto the lifetime ECL:

| Scenario | Unemployment shock | PD multiplier | LGD multiplier | Rationale |
|---|---|---|---|---|
| Base | +0.0 pp | 1.00x | 1.00x | Status quo |
| Adverse | +2.5 pp | 1.50x | 1.10x | Mild recession (2001-style) |
| Severely Adverse | +5.5 pp | 2.50x | 1.20x | 2008-2009 Global Financial Crisis anchor |

### PD uplift rule

A common academic rule of thumb for US consumer credit: **each +1pp in unemployment rate → ~30% increase in PD**. Compounding gives:
- Adverse (+2.5pp): PD × 1.30^2.5 ≈ 1.92x → we round to 1.5x as a conservative overlay
- Severely Adverse (+5.5pp): PD × 1.30^5.5 ≈ 4.35x → we cap at 2.5x to avoid over-stressing

The Severely Adverse multiplier is also benchmarked against actual LendingClub 2008-2009 vintage loss rates, which ran ~2-3× the 2007 baseline.

### LGD stress

Recoveries fall in downturns (collateral values decline, collection becomes harder). We apply a modest LGD uplift as a secondary stress layer.

In [ ]:
# A.11 — Macro scenarios
SCENARIOS = {
    "Base":             {"unemp_shock_pp": 0.0, "pd_mult": 1.00, "lgd_mult": 1.00},
    "Adverse":          {"unemp_shock_pp": 2.5, "pd_mult": 1.50, "lgd_mult": 1.10},
    "Severely Adverse": {"unemp_shock_pp": 5.5, "pd_mult": 2.50, "lgd_mult": 1.20},
}

# Pass 1: compute stressed 12m EL and lifetime ECL for each scenario
raw = {}
for name, p in SCENARIOS.items():
    stress_pd  = (pd12["PD_12m"] * p["pd_mult"]).clip(0, 1)
    stress_lgd = min(LGD_CONST * p["lgd_mult"], 1.0)
    stress_el_12m  = (stress_pd * stress_lgd * pd12["EAD"]).sum()
    stress_life_pd = 1 - (1 - stress_pd).clip(lower=1e-9)**(pd12["term_num"]/12)
    stress_el_life = (stress_life_pd * stress_lgd * pd12["EAD"]).sum()
    raw[name] = {
        "pd_mult": p["pd_mult"],
        "lgd_mult": p["lgd_mult"],
        "unemp_shock_pp": p["unemp_shock_pp"],
        "el_12m":  stress_el_12m,
        "el_life": stress_el_life,
    }

base_life_el = raw["Base"]["el_life"]

# Pass 2: format for display (now we can reference the Base lifetime for vs-base ratios)
rows = []
for name, v in raw.items():
    rows.append({
        "Scenario":           name,
        "Δ Unemp (pp)":       f"+{v['unemp_shock_pp']:.1f}",
        "PD×":                f"{v['pd_mult']:.2f}",
        "LGD×":               f"{v['lgd_mult']:.2f}",
        "12m EL ($M)":        f"{v['el_12m']/1e6:,.2f}",
        "12m loss rate":      f"{v['el_12m']/total_ead:.4%}",
        "Lifetime ECL ($M)":  f"{v['el_life']/1e6:,.2f}",
        "Lifetime rate":      f"{v['el_life']/total_ead:.4%}",
        "vs base (lifetime)": f"{v['el_life']/base_life_el:.2f}x",
    })
stress_df = pd.DataFrame(rows)
print(stress_df.to_string(index=False))

### A.11.1 — Visualization of stressed loss rates

In [ ]:
# A.11.1 — Plot stressed loss rates (reuses raw[] from A.11)
scenarios_list = list(raw.keys())
el_12m_rates  = [raw[s]["el_12m"]  / total_ead * 100 for s in scenarios_list]
el_life_rates = [raw[s]["el_life"] / total_ead * 100 for s in scenarios_list]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(scenarios_list))
w = 0.35
b1 = ax.bar(x - w/2, el_12m_rates,  w, label="12-month loss rate")
b2 = ax.bar(x + w/2, el_life_rates, w, label="Lifetime loss rate")
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h, f"{h:.2f}%",
                ha="center", va="bottom", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(scenarios_list)
ax.set_ylabel("Portfolio loss rate (%)")
ax.set_title("Stressed EL under macro scenarios")
ax.legend()
plt.tight_layout()
plt.show()

## A.12 — LGD distribution on defaulted loans

A point-estimate LGD hides a lot of variance. Here we plot the **empirical distribution** of loss rates across the 570 defaulted loans in our sample. We report two conventions side-by-side:

- **LGD_funded** = realized_loss / funded_amnt (what the backtest uses)
- **LGD_outstanding** ≈ realized_loss / (funded_amnt − total_rec_prncp) (Basel/CECL convention)

The gap between these two distributions is the core reason the backtest's +13% over-reserve is less meaningful than it first appears — it's a denominator artefact.

In [ ]:
# A.12 — LGD distribution
defaulters = pd12[pd12["event"] == 1].copy()
defaulters["lgd_funded"]      = defaulters["realized_loss"] / defaulters["funded_amnt"].clip(lower=1)
outstanding                   = (defaulters["funded_amnt"] - defaulters["total_rec_prncp"].fillna(0)).clip(lower=1)
defaulters["lgd_outstanding"] = (defaulters["realized_loss"] / outstanding).clip(upper=1.0)

print(f"Number of defaulters (event=1): {len(defaulters):,}")
print(f"LGD_funded:      mean {defaulters['lgd_funded'].mean():.4%}  median {defaulters['lgd_funded'].median():.4%}")
print(f"LGD_outstanding: mean {defaulters['lgd_outstanding'].mean():.4%}  median {defaulters['lgd_outstanding'].median():.4%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(defaulters["lgd_funded"], bins=30, edgecolor="black", alpha=0.8)
axes[0].axvline(defaulters["lgd_funded"].mean(), color="red", linestyle="--",
                label=f"mean = {defaulters['lgd_funded'].mean():.2%}")
axes[0].set_title(f"LGD_funded distribution (n={len(defaulters)})")
axes[0].set_xlabel("loss / funded_amnt")
axes[0].legend()

axes[1].hist(defaulters["lgd_outstanding"], bins=30, edgecolor="black", alpha=0.8, color="orange")
axes[1].axvline(defaulters["lgd_outstanding"].mean(), color="red", linestyle="--",
                label=f"mean = {defaulters['lgd_outstanding'].mean():.2%}")
axes[1].set_title(f"LGD_outstanding distribution (n={len(defaulters)})")
axes[1].set_xlabel("loss / outstanding_at_default")
axes[1].legend()
plt.tight_layout()
plt.show()

## A.13 — Export outputs

Write the enriched loan-level table and portfolio summary to `outputs/` for downstream sections (Member B's pricing notebook, Member C's portfolio monitoring, Member D's stress / RAROC / credit playbook).

In [ ]:
# A.13.1 — Loan-level export
export_cols = [
    "id", "issue_d", "vintage", "term_num",
    "grade", "sub_grade", "fico_range_low", "fico_band", "purpose",
    "PD_12m", "LGD", "EAD", "EL_12m", "EL_rate",
    "monthly_hazard", "lifetime_PD", "lifetime_EL",
    "event", "realized_loss",
    "int_rate", "required_rate_logit",   # kept for Member B
]
export_cols = [c for c in export_cols if c in pd12.columns]
loan_out = pd12[export_cols].copy()
loan_out.to_csv(OUT_DIR / "loan_level_EL_and_lifetimeECL.csv", index=False)
print(f"Wrote: {OUT_DIR / 'loan_level_EL_and_lifetimeECL.csv'}  ({len(loan_out):,} rows × {len(export_cols)} cols)")

In [ ]:
# A.13.2 — Portfolio summary export
summary = pd.DataFrame([
    {"metric": "loans",                           "value": len(pd12)},
    {"metric": "EAD_convention",                  "value": "outstanding (funded_amnt - total_rec_prncp)"},
    {"metric": "total_EAD_usd",                   "value": total_ead},
    {"metric": "total_funded_amnt_usd_reference", "value": pd12["funded_amnt"].sum()},
    {"metric": "ead_weighted_PD_12m",             "value": ead_weighted_pd},
    {"metric": "portfolio_loss_rate_12m",         "value": portfolio_rate},
    {"metric": "total_EL_12m_usd",                "value": total_el},
    {"metric": "total_lifetime_ECL_usd",          "value": total_lifetime_el},
    {"metric": "portfolio_loss_rate_lifetime",    "value": lifetime_rate},
    {"metric": "LGD_convention",                  "value": "outstanding (Basel/CECL)"},
    {"metric": "LGD_const_Section1",              "value": LGD_CONST},
    {"metric": "LGD_empirical_2017Q1_outstanding","value": empirical_lgd_outstanding},
    {"metric": "realized_loss_usd_full_sample",   "value": total_realized},
    {"metric": "coverage_ratio_full_sample",      "value": total_el / total_realized},
    {"metric": "coverage_ratio_2017Q1_only",      "value": q1["EL_12m"].sum() / q1["realized_loss"].sum()},
])
summary.to_csv(OUT_DIR / "portfolio_summary_sectionA.csv", index=False)
print(summary.to_string(index=False))
print(f"\nWrote: {OUT_DIR / 'portfolio_summary_sectionA.csv'}")

In [ ]:
# A.13.3 — Vintage backtest export (for Member D's credit playbook)
vintage.to_csv(OUT_DIR / "vintage_backtest_sectionA.csv")
stress_df.to_csv(OUT_DIR / "macro_stress_sectionA.csv", index=False)
print(f"Wrote: {OUT_DIR / 'vintage_backtest_sectionA.csv'}")
print(f"Wrote: {OUT_DIR / 'macro_stress_sectionA.csv'}")

---

# PART B — Pricing & Origination (Member B's scope)

> **These cells are Member B's responsibility.** They were drafted here during early exploration but the deliverables (swap-set analysis, required-rate scatter, underpriced-segment identification) belong in Member B's notebook.
>
> They are **left here for reference** so Member B can lift the code directly. The teammate's `required_rate_logit` column is already in the dataset; the cells below compare it to actual `int_rate`.

## B.1 — Required rate (preliminary calc)

In [ ]:
# B.1 — The teammate's dataset already contains required_rate_logit.
# Member B will extend this with:
#    Required rate = funding cost + op cost + PD×LGD + capital charge + margin
# Here we just inspect the teammate's version.
if "required_rate_logit" in pd12.columns:
    rr = pd12["required_rate_logit"]
    print("required_rate_logit distribution:")
    print(rr.describe().round(4))

## B.2 — Required rate vs actual int_rate

In [ ]:
# B.2 — Compare teammate's required rate vs LC's actual int_rate (scatter)
if "int_rate" in pd12.columns and "required_rate_logit" in pd12.columns:
    fig, ax = plt.subplots(figsize=(7, 7))
    sample = pd12.dropna(subset=["int_rate", "required_rate_logit"]).sample(
        min(3000, len(pd12)), random_state=42)
    # int_rate may be a string like "10.5%" — try numeric cast
    y = pd.to_numeric(sample["int_rate"].astype(str).str.rstrip("%"), errors="coerce") / 100
    ax.scatter(sample["required_rate_logit"], y, alpha=0.2, s=12)
    lim = [0, max(sample["required_rate_logit"].max(), y.max())*1.05]
    ax.plot(lim, lim, "r--", label="y = x (fair pricing)")
    ax.set_xlabel("Required rate (model)")
    ax.set_ylabel("Actual int_rate (LC)")
    ax.set_title("Required vs actual interest rate — Member B's swap-set source plot")
    ax.legend()
    ax.set_xlim(lim); ax.set_ylim(lim)
    plt.tight_layout()
    plt.show()

---

# PART Z — Take-aways for the report narrative

## Z.1 — Headline numbers (for the executive summary)

- **Portfolio EAD:** ~$271M across 17,202 loans
- **Portfolio 12-month loss rate:** ~5.2%
- **Total 12-month EL:** ~$14.2M
- **Total lifetime ECL:** ~$30-40M (depends on constant-hazard extension)
- **Backtest (2017Q1, the only fully-observed cohort):** coverage 1.13x — within CECL's ±15% tolerance band

## Z.2 — Key insight for the report story

> Aggregate backtest appears to over-reserve by +130%. This is **not** a model failure — it is a **right-censoring artefact**: vintages issued after 2017Q2 have not yet been observed for a full 12 months. Restricting the backtest to the only fully-observed cohort (2017Q1) gives a coverage ratio of **1.13x**, which is well within the industry's ±15% tolerance.

## Z.3 — Model limitation discovered via backtest

> Applying `LGD_outstanding = 0.9154` to `funded_amnt` **overstates LGD** by ~40% because `funded_amnt` includes principal that has already been paid down at the time of default. The empirical `LGD_funded ≈ 64%` is the economically correct number for this denominator. The current EL is well-calibrated *only because* this LGD overstatement coincidentally offsets an under-prediction in PD (7.3% actual vs 5.8% predicted for 2017Q1). A production model should reconcile these two errors instead of letting them cancel.

## Z.4 — Segmentation story

> EL concentrates in low-grade (D/E/F/G), long-term (60m), and high-risk-purpose (small_business) buckets. Grade and FICO carry **complementary** risk information: within every grade, the loss rate still falls by 2-4x as FICO rises from subprime to top tier — suggesting LC's internal grade alone does not fully exploit external bureau data.

## Z.5 — Stress-testing story

> Under a 2008-style severely-adverse macro shock (+5.5pp unemployment, PD × 2.5, LGD × 1.2), portfolio lifetime loss rate rises from ~X% to ~Y% — a ~Nx multiplier. This is the capital-planning anchor for Member D's RAROC / credit-playbook section.

## Z.6 — Pointers for next deliverables

- **Member A next steps (if time permits):**
  - Re-derive LGD with both denominators and report as a range, not a point
  - Extend backtest to 2017Q2 (partial observation, still useful for trend)
  - Build vintage-specific seasoning curves instead of constant-hazard extrapolation
- **For Member D (stress & RAROC):** use the `macro_stress_sectionA.csv` export directly; the scenario multipliers are documented in A.11
- **For Member B (pricing):** the scatter in B.2 is the starting point for a swap-set analysis
